In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from psiop_qwen import *

# psiOp quantization

## Kohn–Nirenberg Quantization with FFT

### 1D influence of parameters: freq_windows, clamp_values and space_windows

#### Basic case

In [ ]:
# ============================================================
# 1) 1D periodic grid + FFT frequencies
# ============================================================
L = 10
N = 1024
dx = L / N
x_grid = -L/2 + dx * np.arange(N)
dx = x_grid[1] - x_grid[0]
# FFT frequencies (psipy format)
kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
kx = np.fft.fftshift(kx)

# ============================================================
# 2) Symbol S(x, ξ) in SymPy
# ============================================================
x, xi = symbols('x xi', real=True)
S_expr     = x**2 + xi**2 + 1
S_inv_expr = 1/(x**2 + xi**2 + 1 + 1e-12)   # stabilized

# ============================================================
# 3) Operators ψOp(S) and ψOp(S⁻¹)
# ============================================================
OpS     = PseudoDifferentialOperator(expr=S_expr,     vars_x=[x], mode='symbol')
OpS_inv = PseudoDifferentialOperator(expr=S_inv_expr, vars_x=[x], mode='symbol')

# ============================================================
# 4) Test function
# ============================================================
f_exact = np.exp(-x_grid**2)
g_exact = (-3*x_grid**2 + 3) * f_exact

# ============================================================
# 5) KN parameters
# ============================================================
freq_windows = [None, 'gaussian', 'hann']
clamp_values = [1e2, 1e4, 1e6]
space_windows = [False, True]
combinations = list(product(freq_windows, clamp_values, space_windows))
results = []

# ============================================================
# 6) Main loop
# ============================================================
for fw, clamp, sw in combinations:
    label = f"fw={fw}, clamp={clamp:.0e}, sw={sw}"
    # --------------------------------------------------------
    # ψOp(S)[f] 
    # --------------------------------------------------------
    g_approx = OpS.apply(
        f_exact,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='periodic',
        y_grid=None,
        ky=None,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_forward = np.linalg.norm(g_exact - g_approx) / np.linalg.norm(g_exact)
    # --------------------------------------------------------
    # ψOp(S⁻¹)[g]
    # --------------------------------------------------------
    f_approx = OpS_inv.apply(
        g_approx,
        x_grid,
        kx,
        boundary_condition='periodic',
        y_grid=None,
        ky=None,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_inverse = np.linalg.norm(f_exact - f_approx) / np.linalg.norm(f_exact)
    results.append((fw, clamp, sw, err_forward, err_inverse))
    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(x_grid, g_exact, lw=2, label='g_exact')
    plt.plot(x_grid, g_approx, '--', lw=2, label='ψOp(S)[f]')
    plt.title(f"Forward {label}\nerr={err_forward:.2e}")
    plt.grid(True); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(x_grid, f_exact, lw=2, label='f_exact')
    plt.plot(x_grid, f_approx, '--', lw=2, label='ψOp(S⁻¹)[g]')
    plt.title(f"Inverse {label}\nerr={err_inverse:.2e}")
    plt.grid(True); plt.legend()
    plt.suptitle("PseudoDifferentialOperator — Kohn–Nirenberg Test (1D)")
    plt.tight_layout()
    plt.show()

# ============================================================
# 7) Summary
# ============================================================
print("\nSummary of Relative Errors:")
print(f"{'FreqWin':>10} {'Clamp':>10} {'SpaceWin':>10} {'ErrFwd':>12} {'ErrInv':>12}")
for fw, clamp, sw, e1, e2 in results:
    print(f"{str(fw):>10} {clamp:10.0e} {str(sw):>10} {e1:12.2e} {e2:12.2e}")


#### Advanced case

In [ ]:
# ============================================================
# 1) 1D periodic grid + FFT frequencies
# ============================================================
L = 10
N = 1024
dx = L / N
x_grid = -L/2 + dx * np.arange(N)
dx = x_grid[1] - x_grid[0]
# FFT frequencies (psipy format)
kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
kx = np.fft.fftshift(kx)

# ============================================================
# 2) Symbol S(x, ξ) in SymPy
# ============================================================
x, xi = symbols('x xi', real=True)
S_expr     = (1 + 0.5 * sin(3 * x)) * xi**2
S_inv_expr = 1.0 / (S_expr + 1e-6)   # stabilized

# ============================================================
# 3) Operators ψOp(S) and ψOp(S⁻¹)
# ============================================================
OpS     = PseudoDifferentialOperator(expr=S_expr,     vars_x=[x], mode='symbol')
OpS_inv = PseudoDifferentialOperator(expr=S_inv_expr, vars_x=[x], mode='symbol')

# ============================================================
# 4) Test function
# ============================================================
f_exact = f_exact = np.exp(-x_grid**2) * np.cos(10 * x_grid)

# ============================================================
# 5) KN parameters
# ============================================================
freq_windows = [None, 'gaussian', 'hann']
clamp_values = [1e2, 1e4, 1e6]
space_windows = [False, True]
combinations = list(product(freq_windows, clamp_values, space_windows))
results = []

# ============================================================
# 6) Main loop
# ============================================================
for fw, clamp, sw in combinations:
    label = f"fw={fw}, clamp={clamp:.0e}, sw={sw}"
    # --------------------------------------------------------
    # ψOp(S)[f] 
    # --------------------------------------------------------
    g_approx = OpS.apply(
        f_exact,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='periodic',
        y_grid=None,
        ky=None,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    # --- Compute amplification ---
    ampl_ratio = np.linalg.norm(g_approx) / np.linalg.norm(f_exact)
    # --------------------------------------------------------
    # ψOp(S⁻¹)[g]
    # --------------------------------------------------------
    f_approx = OpS_inv.apply(
        g_approx,
        x_grid,
        kx,
        boundary_condition='periodic',
        y_grid=None,
        ky=None,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    # --- Compute inverse error ---
    err_inverse = np.linalg.norm(f_exact - f_approx) / np.linalg.norm(f_exact)

    # --- Save results ---
    results.append((fw, clamp, sw, ampl_ratio, err_inverse))

    # --- Plot results for this configuration ---
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(x_grid, g_approx.real, '--', label='ψOp(S)[f]', lw=2)
    plt.title(f"Forward: {label}\n‖g‖/‖f‖ = {ampl_ratio:.2e}")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(x_grid, f_exact, label='f_exact', lw=2)
    plt.plot(x_grid, f_approx.real, '--', label='ψOp(S⁻¹)[g]', lw=2)
    plt.title(f"Inverse: {label}\nerr={err_inverse:.2e}")
    plt.grid(True)
    plt.legend()

    plt.suptitle("Kohn–Nirenberg 1D Evaluation")
    plt.tight_layout()
    plt.show()

# --- Summary Table ---
print("\nSummary of Amplification and Inversion Errors:")
print(f"{'FreqWin':>10} {'Clamp':>10} {'SpaceWin':>10} {'‖g‖/‖f‖':>12} {'ErrInv':>12}")
for fw, clamp, sw, ampl, err2 in results:
    print(f"{str(fw):>10} {clamp:10.0e} {str(sw):>10} {ampl:12.2e} {err2:12.2e}")


### 2D influence of parameters: freq_windows, clamp_values and space_windows

#### Basic case

In [ ]:
# ============================================================
# 1) 2D periodic grid + FFT frequencies
# ============================================================
L = 10
N = 64
dx = L / N
x_grid = -L/2 + dx * np.arange(N)
y_grid = -L/2 + dx * np.arange(N)
dx = x_grid[1] - x_grid[0]
dy = y_grid[1] - y_grid[0]
# FFT frequencies (psipy format)
kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
kx = np.fft.fftshift(kx)
ky = 2 * np.pi * np.fft.fftfreq(N, d=dy)
ky = np.fft.fftshift(ky)
# --- 2D grids ---
X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')

# ============================================================
# 2) Symbol S(x, y, ξ, η) in SymPy
# ============================================================
x, y, xi, eta = symbols('x y xi eta', real=True)
u = Function('u')(x, y)
# --- Definition of the symbol and its inverse ---
S_expr     = x**2 + y**2 + xi**2 + eta**2 + 1
S_inv_expr = 1.0 / (S_expr + 1e-10)

# ============================================================
# 3) Operators ψOp(S) and ψOp(S⁻¹)
# ============================================================
OpS     = PseudoDifferentialOperator(expr=S_expr,     vars_x=[x, y], mode='symbol')
OpS_inv = PseudoDifferentialOperator(expr=S_inv_expr, vars_x=[x, y], mode='symbol')

# ============================================================
# 4) Test function
# ============================================================
# --- Test function: Centered Gaussian ---
f_exact = np.exp(-(X**2 + Y**2))
# --- Target output: Op(p)[f] = (-3 x² - 3 y² + 5) * f_exact ---
g_exact = (-3 * X**2 - 3 * Y**2 + 5) * f_exact

# ============================================================
# 5) KN parameters
# ============================================================
freq_windows = [None, 'gaussian', 'hann']
clamp_values = [1e2, 1e4, 1e6]
space_windows = [False, True]
combinations = list(product(freq_windows, clamp_values, space_windows))
results = []

# ============================================================
# 6) Main loop
# ============================================================
for fw, clamp, sw in combinations:
    label = f"fw={fw}, clamp={clamp:.0e}, sw={sw}"
    # --------------------------------------------------------
    # ψOp(S)[f]
    # --------------------------------------------------------
    g_approx = OpS.apply(
        f_exact,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='periodic',
        y_grid=y_grid,
        ky=ky,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_forward = np.linalg.norm(g_exact - g_approx) / np.linalg.norm(g_exact)
    # --------------------------------------------------------
    # ψOp(S⁻¹)[g]
    # --------------------------------------------------------
    f_approx = OpS_inv.apply(
        g_approx,
        x_grid,
        kx,
        boundary_condition='periodic',
        y_grid=y_grid,
        ky=ky,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_inverse = np.linalg.norm(f_exact - f_approx) / np.linalg.norm(f_exact)
    results.append((fw, clamp, sw, err_forward, err_inverse))
    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    extent = [-L/2, L/2, -L/2, L/2]
    im = axes[0, 0].imshow(f_exact, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[0, 0])
    axes[0, 0].set_title("Exact Input f(x,y)")
    im = axes[0, 1].imshow(g_exact, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[0, 1])
    axes[0, 1].set_title(f"Exact Output g(x,y)\n{label}")
    im = axes[1, 0].imshow(g_approx.real, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[1, 0])
    axes[1, 0].set_title(f"Approx Output ψOp(S)[f]\nerr={err_forward:.2e}")
    im = axes[1, 1].imshow(f_approx.real, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[1, 1])
    axes[1, 1].set_title(f"Reconstructed f ≈ ψOp(S⁻¹)[g]\nerr={err_inverse:.2e}")
    plt.suptitle("2D Kohn–Nirenberg Evaluation")
    plt.tight_layout()
    plt.show()

# ============================================================
# 7) Summary
# ============================================================
print("\nSummary of Relative Errors:")
print(f"{'FreqWin':>10} {'Clamp':>10} {'SpaceWin':>10} {'ErrFwd':>12} {'ErrInv':>12}")
for fw, clamp, sw, e1, e2 in results:
    print(f"{str(fw):>10} {clamp:10.0e} {str(sw):>10} {e1:12.2e} {e2:12.2e}")


#### Advanced case

In [ ]:
# ============================================================
# 1) 2D periodic grid + FFT frequencies
# ============================================================
L = 10
N = 64
dx = L / N
x_grid = -L/2 + dx * np.arange(N)
y_grid = -L/2 + dx * np.arange(N)
dx = x_grid[1] - x_grid[0]
dy = y_grid[1] - y_grid[0]
# --- FFT frequencies (psipy format) ---
kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
kx = np.fft.fftshift(kx)
ky = 2 * np.pi * np.fft.fftfreq(N, d=dy)
ky = np.fft.fftshift(ky)
# --- 2D grids ---
X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')

# ============================================================
# 2) Symbol S(x, y, ξ, η) in SymPy
# ============================================================
x, y, xi, eta = symbols('x y xi eta', real=True)
u = Function('u')(x, y)
# --- Definition of the symbol and its inverse ---
S_expr     = (1 + 0.5 * cos(3*x) * sin(2*y)) * (xi**2 + eta**2)
S_inv_expr = 1 / (S_expr + 1e-6)

# ============================================================
# 3) Operators ψOp(S) and ψOp(S⁻¹)
# ============================================================
p_quantum_lens     = PseudoDifferentialOperator(expr=S_expr,     vars_x=[x, y], mode='symbol')
p_quantum_lens_inv = PseudoDifferentialOperator(expr=S_inv_expr, vars_x=[x, y], mode='symbol')

# ============================================================
# 4) Test function
# ============================================================
# --- Test function: oscillating Gaussian (modulated laser beam) ---
f_exact = np.exp(-(X**2 + Y**2)) * np.cos(10 * X)

# ============================================================
# 5) KN parameters
# ============================================================
freq_windows = [None, 'gaussian', 'hann']
space_windows = [False, True]
clamp = 1e4  # Fixed clamp to avoid instability
combinations = list(product(freq_windows, space_windows))
results = []

# ============================================================
# 6) Main loop
# ============================================================
for fw, sw in combinations:
    label = f"fw={fw}, sw={sw}"
    # --------------------------------------------------------
    # ψOp(S)[f]
    # --------------------------------------------------------
    g_approx = p_quantum_lens.apply(
        f_exact,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='periodic',
        y_grid=y_grid,
        ky=ky,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    # --- Compute centered spectra ---
    F_spec = np.log1p(np.abs(fftshift(fft2(f_exact))))
    G_spec = np.log1p(np.abs(fftshift(fft2(g_approx))))
    # --- Frequency axes for plotting ---
    freq_extent = [-N/2, N/2, -N/2, N/2]
    # --- Plotting ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(F_spec, extent=freq_extent, origin='lower', cmap='plasma')
    axes[0].set_title("FFT of f_exact")
    axes[0].set_xlabel("ξ")
    axes[0].set_ylabel("η")
    axes[1].imshow(G_spec, extent=freq_extent, origin='lower', cmap='plasma')
    axes[1].set_title("FFT of g_approx = ψOp(p)[f]")
    axes[1].set_xlabel("ξ")
    axes[1].set_ylabel("η")
    plt.suptitle("Frequency content: input vs amplified output")
    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------
    # ψOp(S⁻¹)[g]
    # --------------------------------------------------------
    f_recon = p_quantum_lens_inv.apply(
        g_approx,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='periodic',
        y_grid=y_grid,
        ky=ky,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_forward = np.linalg.norm(g_approx) / np.linalg.norm(f_exact)
    err_inverse = np.linalg.norm(f_exact - f_recon) / np.linalg.norm(f_exact)
    results.append((fw, sw, err_forward, err_inverse))

    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    extent = [-L/2, L/2, -L/2, L/2]
    im = axes[0, 0].imshow(f_exact, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[0, 0])
    axes[0, 0].set_title("Input: oscillating Gaussian")
    im = axes[0, 1].imshow(g_approx.real, extent=extent, origin='lower', cmap='magma')
    plt.colorbar(im, ax=axes[0, 1])
    axes[0, 1].set_title(f"ψOp(p)[f] {label}\n(norm={err_forward:.2e})")
    im = axes[1, 0].imshow(f_recon.real, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[1, 0])
    axes[1, 0].set_title(f"Reconstructed f ≈ ψOp(p⁻¹)[g]")
    im = axes[1, 1].imshow((f_recon - f_exact).real, extent=extent, origin='lower', cmap='inferno')
    plt.colorbar(im, ax=axes[1, 1])
    axes[1, 1].set_title(f"Error f - f_recon\n(err={err_inverse:.2e})")
    plt.suptitle(f"Kohn–Nirenberg Diffraction Test\n{label}")
    plt.tight_layout()
    plt.show()

# ============================================================
# 7) Summary
# ============================================================
print("\nSummary of Relative Errors:")
print(f"{'FreqWin':>10} {'SpaceWin':>10} {'ErrFwd':>12} {'ErrInv':>12}")
for fw, sw, err1, err2 in results:
    print(f"{str(fw):>10} {str(sw):>10} {err1:12.2e} {err2:12.2e}")


## Non-periodic Kohn–Nirenberg Quantization 

### 1D

#### Basic case

In [ ]:
# ============================================================
# 1) 1D periodic grid + FFT frequencies
# ============================================================
L = 10
N = 1024
dx = L / N
x_grid = -L/2 + dx * np.arange(N)
dx = x_grid[1] - x_grid[0]
# FFT frequencies (psipy format)
kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
kx = np.fft.fftshift(kx)

# ============================================================
# 2) Symbol S(x, ξ) in SymPy
# ============================================================
x, xi = symbols('x xi', real=True)
S_expr     = x**2 + xi**2 + 1
S_inv_expr = 1/(x**2 + xi**2 + 1 + 1e-12)   # stabilized

# ============================================================
# 3) Operators ψOp(S) and ψOp(S⁻¹)
# ============================================================
OpS     = PseudoDifferentialOperator(expr=S_expr,     vars_x=[x], mode='symbol')
OpS_inv = PseudoDifferentialOperator(expr=S_inv_expr, vars_x=[x], mode='symbol')

# ============================================================
# 4) Test function
# ============================================================
f_exact = np.exp(-x_grid**2)
g_exact = (-3*x_grid**2 + 3) * f_exact

# ============================================================
# 5) KN parameters
# ============================================================
freq_windows = [None, 'gaussian', 'hann']
clamp_values = [1e2, 1e4, 1e6]
space_windows = [False, True]
combinations = list(product(freq_windows, clamp_values, space_windows))
results = []

# ============================================================
# 6) Main loop
# ============================================================
for fw, clamp, sw in combinations:
    label = f"fw={fw}, clamp={clamp:.0e}, sw={sw}"
    # --------------------------------------------------------
    # ψOp(S)[f] 
    # --------------------------------------------------------
    g_approx = OpS.apply(
        f_exact,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='dirichlet',
        y_grid=None,
        ky=None,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_forward = np.linalg.norm(g_exact - g_approx) / np.linalg.norm(g_exact)
    # --------------------------------------------------------
    # ψOp(S⁻¹)[g]
    # --------------------------------------------------------
    f_approx = OpS_inv.apply(
        g_approx,
        x_grid,
        kx,
        boundary_condition='dirichlet',
        y_grid=None,
        ky=None,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_inverse = np.linalg.norm(f_exact - f_approx) / np.linalg.norm(f_exact)
    results.append((fw, clamp, sw, err_forward, err_inverse))
    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(x_grid, g_exact, lw=2, label='g_exact')
    plt.plot(x_grid, g_approx, '--', lw=2, label='ψOp(S)[f]')
    plt.title(f"Forward {label}\nerr={err_forward:.2e}")
    plt.grid(True); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(x_grid, f_exact, lw=2, label='f_exact')
    plt.plot(x_grid, f_approx, '--', lw=2, label='ψOp(S⁻¹)[g]')
    plt.title(f"Inverse {label}\nerr={err_inverse:.2e}")
    plt.grid(True); plt.legend()
    plt.suptitle("PseudoDifferentialOperator — Kohn–Nirenberg Test (1D)")
    plt.tight_layout()
    plt.show()

# ============================================================
# 7) Summary
# ============================================================
print("\nSummary of Relative Errors:")
print(f"{'FreqWin':>10} {'Clamp':>10} {'SpaceWin':>10} {'ErrFwd':>12} {'ErrInv':>12}")
for fw, clamp, sw, e1, e2 in results:
    print(f"{str(fw):>10} {clamp:10.0e} {str(sw):>10} {e1:12.2e} {e2:12.2e}")


#### Advanced case

In [ ]:
# ============================================================
# 1) 1D periodic grid + FFT frequencies
# ============================================================
L = 10
N = 1024
dx = L / N
x_grid = -L/2 + dx * np.arange(N)
dx = x_grid[1] - x_grid[0]
# FFT frequencies (psipy format)
kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
kx = np.fft.fftshift(kx)

# ============================================================
# 2) Symbol S(x, ξ) in SymPy
# ============================================================
x, xi = symbols('x xi', real=True)
S_expr     = (1 + 0.5 * sin(3 * x)) * xi**2
S_inv_expr = 1.0 / (S_expr + 1e-6)   # stabilized

# ============================================================
# 3) Operators ψOp(S) and ψOp(S⁻¹)
# ============================================================
OpS     = PseudoDifferentialOperator(expr=S_expr,     vars_x=[x], mode='symbol')
OpS_inv = PseudoDifferentialOperator(expr=S_inv_expr, vars_x=[x], mode='symbol')

# ============================================================
# 4) Test function
# ============================================================
f_exact = f_exact = np.exp(-x_grid**2) * np.cos(10 * x_grid)

# ============================================================
# 5) KN parameters
# ============================================================
freq_windows = [None, 'gaussian', 'hann']
clamp_values = [1e2, 1e4, 1e6]
space_windows = [False, True]
combinations = list(product(freq_windows, clamp_values, space_windows))
results = []

# ============================================================
# 6) Main loop
# ============================================================
for fw, clamp, sw in combinations:
    label = f"fw={fw}, clamp={clamp:.0e}, sw={sw}"
    # --------------------------------------------------------
    # ψOp(S)[f] 
    # --------------------------------------------------------
    g_approx = OpS.apply(
        f_exact,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='dirichlet',
        y_grid=None,
        ky=None,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    # --- Compute amplification ---
    ampl_ratio = np.linalg.norm(g_approx) / np.linalg.norm(f_exact)
    # --------------------------------------------------------
    # ψOp(S⁻¹)[g]
    # --------------------------------------------------------
    f_approx = OpS_inv.apply(
        g_approx,
        x_grid,
        kx,
        boundary_condition='dirichlet',
        y_grid=None,
        ky=None,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    # --- Compute inverse error ---
    err_inverse = np.linalg.norm(f_exact - f_approx) / np.linalg.norm(f_exact)

    # --- Save results ---
    results.append((fw, clamp, sw, ampl_ratio, err_inverse))

    # --- Plot results for this configuration ---
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(x_grid, g_approx.real, '--', label='ψOp(S)[f]', lw=2)
    plt.title(f"Forward: {label}\n‖g‖/‖f‖ = {ampl_ratio:.2e}")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(x_grid, f_exact, label='f_exact', lw=2)
    plt.plot(x_grid, f_approx.real, '--', label='ψOp(S⁻¹)[g]', lw=2)
    plt.title(f"Inverse: {label}\nerr={err_inverse:.2e}")
    plt.grid(True)
    plt.legend()

    plt.suptitle("Kohn–Nirenberg 1D Evaluation")
    plt.tight_layout()
    plt.show()

# --- Summary Table ---
print("\nSummary of Amplification and Inversion Errors:")
print(f"{'FreqWin':>10} {'Clamp':>10} {'SpaceWin':>10} {'‖g‖/‖f‖':>12} {'ErrInv':>12}")
for fw, clamp, sw, ampl, err2 in results:
    print(f"{str(fw):>10} {clamp:10.0e} {str(sw):>10} {ampl:12.2e} {err2:12.2e}")


### 2D

#### Basic case

In [ ]:
# ============================================================
# 1) 2D periodic grid + FFT frequencies
# ============================================================
L = 10
N = 64
dx = L / N
x_grid = -L/2 + dx * np.arange(N)
y_grid = -L/2 + dx * np.arange(N)
dx = x_grid[1] - x_grid[0]
dy = y_grid[1] - y_grid[0]
# FFT frequencies (psipy format)
kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
kx = np.fft.fftshift(kx)
ky = 2 * np.pi * np.fft.fftfreq(N, d=dy)
ky = np.fft.fftshift(ky)
# --- 2D grids ---
X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')

# ============================================================
# 2) Symbol S(x, y, ξ, η) in SymPy
# ============================================================
x, y, xi, eta = symbols('x y xi eta', real=True)
u = Function('u')(x, y)
# --- Definition of the symbol and its inverse ---
S_expr     = x**2 + y**2 + xi**2 + eta**2 + 1
S_inv_expr = 1.0 / (S_expr + 1e-10)

# ============================================================
# 3) Operators ψOp(S) and ψOp(S⁻¹)
# ============================================================
OpS     = PseudoDifferentialOperator(expr=S_expr,     vars_x=[x, y], mode='symbol')
OpS_inv = PseudoDifferentialOperator(expr=S_inv_expr, vars_x=[x, y], mode='symbol')

# ============================================================
# 4) Test function
# ============================================================
# --- Test function: Centered Gaussian ---
f_exact = np.exp(-(X**2 + Y**2))
# --- Target output: Op(p)[f] = (-3 x² - 3 y² + 5) * f_exact ---
g_exact = (-3 * X**2 - 3 * Y**2 + 5) * f_exact

# ============================================================
# 5) KN parameters
# ============================================================
freq_windows = [None, 'gaussian', 'hann']
clamp_values = [1e2, 1e4, 1e6]
space_windows = [False, True]
combinations = list(product(freq_windows, clamp_values, space_windows))
results = []

# ============================================================
# 6) Main loop
# ============================================================
for fw, clamp, sw in combinations:
    label = f"fw={fw}, clamp={clamp:.0e}, sw={sw}"
    # --------------------------------------------------------
    # ψOp(S)[f]
    # --------------------------------------------------------
    g_approx = OpS.apply(
        f_exact,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='dirichlet',
        y_grid=y_grid,
        ky=ky,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_forward = np.linalg.norm(g_exact - g_approx) / np.linalg.norm(g_exact)
    # --------------------------------------------------------
    # ψOp(S⁻¹)[g]
    # --------------------------------------------------------
    f_approx = OpS_inv.apply(
        g_approx,
        x_grid,
        kx,
        boundary_condition='dirichlet',
        y_grid=y_grid,
        ky=ky,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_inverse = np.linalg.norm(f_exact - f_approx) / np.linalg.norm(f_exact)
    results.append((fw, clamp, sw, err_forward, err_inverse))
    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    extent = [-L/2, L/2, -L/2, L/2]
    im = axes[0, 0].imshow(f_exact, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[0, 0])
    axes[0, 0].set_title("Exact Input f(x,y)")
    im = axes[0, 1].imshow(g_exact, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[0, 1])
    axes[0, 1].set_title(f"Exact Output g(x,y)\n{label}")
    im = axes[1, 0].imshow(g_approx.real, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[1, 0])
    axes[1, 0].set_title(f"Approx Output ψOp(S)[f]\nerr={err_forward:.2e}")
    im = axes[1, 1].imshow(f_approx.real, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[1, 1])
    axes[1, 1].set_title(f"Reconstructed f ≈ ψOp(S⁻¹)[g]\nerr={err_inverse:.2e}")
    plt.suptitle("2D Kohn–Nirenberg Evaluation")
    plt.tight_layout()
    plt.show()

# ============================================================
# 7) Summary
# ============================================================
print("\nSummary of Relative Errors:")
print(f"{'FreqWin':>10} {'Clamp':>10} {'SpaceWin':>10} {'ErrFwd':>12} {'ErrInv':>12}")
for fw, clamp, sw, e1, e2 in results:
    print(f"{str(fw):>10} {clamp:10.0e} {str(sw):>10} {e1:12.2e} {e2:12.2e}")


#### Advanced case

In [ ]:
# ============================================================
# 1) 2D periodic grid + FFT frequencies
# ============================================================
L = 10
N = 64
dx = L / N
x_grid = -L/2 + dx * np.arange(N)
y_grid = -L/2 + dx * np.arange(N)
dx = x_grid[1] - x_grid[0]
dy = y_grid[1] - y_grid[0]
# --- FFT frequencies (psipy format) ---
kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
kx = np.fft.fftshift(kx)
ky = 2 * np.pi * np.fft.fftfreq(N, d=dy)
ky = np.fft.fftshift(ky)
# --- 2D grids ---
X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')

# ============================================================
# 2) Symbol S(x, y, ξ, η) in SymPy
# ============================================================
x, y, xi, eta = symbols('x y xi eta', real=True)
u = Function('u')(x, y)
# --- Definition of the symbol and its inverse ---
S_expr     = (1 + 0.5 * cos(3*x) * sin(2*y)) * (xi**2 + eta**2)
S_inv_expr = 1 / (S_expr + 1e-6)

# ============================================================
# 3) Operators ψOp(S) and ψOp(S⁻¹)
# ============================================================
p_quantum_lens     = PseudoDifferentialOperator(expr=S_expr,     vars_x=[x, y], mode='symbol')
p_quantum_lens_inv = PseudoDifferentialOperator(expr=S_inv_expr, vars_x=[x, y], mode='symbol')

# ============================================================
# 4) Test function
# ============================================================
# --- Test function: oscillating Gaussian (modulated laser beam) ---
f_exact = np.exp(-(X**2 + Y**2)) * np.cos(10 * X)

# ============================================================
# 5) KN parameters
# ============================================================
freq_windows = [None, 'gaussian', 'hann']
space_windows = [False, True]
clamp = 1e4  # Fixed clamp to avoid instability
combinations = list(product(freq_windows, space_windows))
results = []

# ============================================================
# 6) Main loop
# ============================================================
for fw, sw in combinations:
    label = f"fw={fw}, sw={sw}"
    # --------------------------------------------------------
    # ψOp(S)[f]
    # --------------------------------------------------------
    g_approx = p_quantum_lens.apply(
        f_exact,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='dirichlet',
        y_grid=y_grid,
        ky=ky,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    # --- Compute centered spectra ---
    F_spec = np.log1p(np.abs(fftshift(fft2(f_exact))))
    G_spec = np.log1p(np.abs(fftshift(fft2(g_approx))))
    # --- Frequency axes for plotting ---
    freq_extent = [-N/2, N/2, -N/2, N/2]
    # --- Plotting ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(F_spec, extent=freq_extent, origin='lower', cmap='plasma')
    axes[0].set_title("FFT of f_exact")
    axes[0].set_xlabel("ξ")
    axes[0].set_ylabel("η")
    axes[1].imshow(G_spec, extent=freq_extent, origin='lower', cmap='plasma')
    axes[1].set_title("FFT of g_approx = ψOp(p)[f]")
    axes[1].set_xlabel("ξ")
    axes[1].set_ylabel("η")
    plt.suptitle("Frequency content: input vs amplified output")
    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------
    # ψOp(S⁻¹)[g]
    # --------------------------------------------------------
    f_recon = p_quantum_lens_inv.apply(
        g_approx,
        x_grid,        # ← 2nd positional parameter
        kx,            # ← 3rd positional parameter
        boundary_condition='dirichlet',
        y_grid=y_grid,
        ky=ky,
        dealiasing_mask=None,
        freq_window=fw,
        clamp=clamp,
        space_window=sw
    )
    err_forward = np.linalg.norm(g_approx) / np.linalg.norm(f_exact)
    err_inverse = np.linalg.norm(f_exact - f_recon) / np.linalg.norm(f_exact)
    results.append((fw, sw, err_forward, err_inverse))

    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    extent = [-L/2, L/2, -L/2, L/2]
    im = axes[0, 0].imshow(f_exact, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[0, 0])
    axes[0, 0].set_title("Input: oscillating Gaussian")
    im = axes[0, 1].imshow(g_approx.real, extent=extent, origin='lower', cmap='magma')
    plt.colorbar(im, ax=axes[0, 1])
    axes[0, 1].set_title(f"ψOp(p)[f] {label}\n(norm={err_forward:.2e})")
    im = axes[1, 0].imshow(f_recon.real, extent=extent, origin='lower', cmap='viridis')
    plt.colorbar(im, ax=axes[1, 0])
    axes[1, 0].set_title(f"Reconstructed f ≈ ψOp(p⁻¹)[g]")
    im = axes[1, 1].imshow((f_recon - f_exact).real, extent=extent, origin='lower', cmap='inferno')
    plt.colorbar(im, ax=axes[1, 1])
    axes[1, 1].set_title(f"Error f - f_recon\n(err={err_inverse:.2e})")
    plt.suptitle(f"Kohn–Nirenberg Diffraction Test\n{label}")
    plt.tight_layout()
    plt.show()

# ============================================================
# 7) Summary
# ============================================================
print("\nSummary of Relative Errors:")
print(f"{'FreqWin':>10} {'SpaceWin':>10} {'ErrFwd':>12} {'ErrInv':>12}")
for fw, sw, err1, err2 in results:
    print(f"{str(fw):>10} {str(sw):>10} {err1:12.2e} {err2:12.2e}")
